# CS383: Data Science and Machine Learning
## Lecture 2 Exercises — Python Refresher, NumPy, Vectorized Computing

**Make a copy of this notebook before you start** (do not edit this original). Fill in every `__________` blank, then run all cells top to bottom before submitting. This notebook is graded with Otter Grader — do not delete or modify the setup cells.

In [ ]:
import numpy as np

---

## Exercise 1 — Vectorized Math vs. Python Loops

In this lab you'll write both a loop version and a vectorized version of the same calculation, time them yourself, and see the difference firsthand.

### Scenario
You have exam scores for a large class and want to compute each student's *squared deviation* from the class average (`(score - mean_score) ** 2`) — the same building block used to compute variance and standard deviation, which you'll see formalized later in the course.

In [ ]:
rng = np.random.default_rng(383)
scores = rng.normal(loc=75, scale=10, size=200_000).round(1)
scores = np.clip(scores, 0, 100)   # keep scores in a valid 0-100 range

print(scores[:10])
print(len(scores))

### Step 1 — Loop version

In [ ]:
import time

start = time.time()
mean_score = sum(scores) / len(scores)
squared_devs_loop = []
for s in scores:
    squared_devs_loop.append((s - mean_score) ** 2)
loop_time = time.time() - start

print(f"Loop version took {loop_time:.3f} seconds")
print(squared_devs_loop[:5])

### Step 2 — Your turn: vectorize it

Fill in the two blanks so `squared_devs_vectorized` computes the same thing as `squared_devs_loop`, without writing a Python `for` loop.

In [ ]:
start = time.time()
mean_score_np = scores.mean()
squared_devs_vectorized = (__________ - __________) ** 2
vectorized_time = time.time() - start

print(f"Vectorized version took {vectorized_time:.5f} seconds")
print(squared_devs_vectorized[:5])
print(f"Speedup: {loop_time / vectorized_time:,.0f}x")
print(f"Results match: {np.allclose(squared_devs_loop, squared_devs_vectorized)}")

### Reflect
- How much faster was your vectorized version?
- What would happen to the loop version's time if you doubled `size` to 400,000 in the data cell above? Try it and see if your prediction was right.

---

## Exercise 2 — Basic Data Calculations, on Real Data

Same idea as Part 5, now applied to the live NYC 311 dataset from Lecture 1.

In [ ]:
import requests

SOCRATA_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

try:
    response = requests.get(
        SOCRATA_URL,
        params={
            "$limit": 20000,
            "$order": "created_date DESC",
            "$select": "complaint_type,borough",
        },
        timeout=8,
    )
    response.raise_for_status()
    rows = response.json()
    complaint_arr = np.array([r.get("complaint_type", "UNKNOWN") for r in rows])
    borough_arr = np.array([r.get("borough", "UNKNOWN") for r in rows])
    live = True
except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    complaint_arr = rng.choice(
        complaint_types, size=20000,
        p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
    )
    borough_arr = rng.choice(boroughs, size=20000, p=[0.22, 0.32, 0.26, 0.16, 0.04])
    live = False

print(f"{'Live' if live else 'Offline fallback'} data: {len(complaint_arr):,} 311 records")
print(complaint_arr[:5])

### Counting, the loop way

In [ ]:
target = "Noise - Residential"

start = time.time()
count_loop = 0
for c in complaint_arr:
    if c == target:
        count_loop += 1
loop_time = time.time() - start

print(f"'{target}' appeared {count_loop:,} times")
print(f"Loop took {loop_time:.4f} seconds")

### Your turn: count the vectorized way

Fill in the blank so `count_vectorized` counts the same thing as `count_loop`, using a boolean mask and `np.sum` instead of a loop.

In [ ]:
start = time.time()
count_vectorized = np.sum(__________ == target)
vectorized_time = time.time() - start

print(f"Vectorized count: {count_vectorized:,}")
print(f"Vectorized took {vectorized_time:.6f} seconds")
print(f"Match: {count_loop == count_vectorized}")

### Basic summary calculations, fully vectorized

In [ ]:
boroughs_unique, borough_counts = np.unique(borough_arr, return_counts=True)
borough_percentages = borough_counts / borough_counts.sum() * 100

order = np.argsort(-borough_counts)
for b, count, pct in zip(boroughs_unique[order], borough_counts[order], borough_percentages[order]):
    print(f"{b:15s} {count:6,d} complaints  ({pct:4.1f}%)")

`np.unique(..., return_counts=True)` is itself a vectorized operation — no loop required to tally every borough's complaint count.

---

## Exercise 3 — Reflection (Exit Ticket)

Answer the following in your own words.

1. What does it mean for a NumPy operation to be "vectorized"?
2. Why was the vectorized version of the exam-score calculation faster than the loop version?
3. What does `arr[arr > 20]` do, step by step?
4. Give one example (from today or otherwise) of a calculation where you'd still reach for a loop.
5. What question do you still have about NumPy or vectorization before Lecture 3?

**Your responses:**

1.  
2.  
3.  
4.  
5.  

## Optional Challenge

Pick any numeric data you care about (school grades, sports stats, workout logs, game scores, etc.). You'll build the full loop-vs-vectorized comparison one step at a time.

### Step 1 — Pick your data, write the loop version

In [ ]:
# At least 5 numeric values
my_data = [__________, __________, __________, __________, __________]

# Loop-based calculation: a sum, an average, or a count of values meeting some condition
# Your code here


### Step 2 — Rewrite it as a vectorized NumPy calculation

In [ ]:
# Convert my_data to a NumPy array, then redo Step 1's calculation without a loop
# Your code here


### Step 3 — Scale up and time both versions

In [ ]:
# Generate a much larger, randomly generated version of your data (at least 100,000 values)
# Time both the loop version and the vectorized version, and report the speedup
# Your code here


### Big idea
> The habit of asking "can this be vectorized?" instead of reaching for a loop first is one of the most valuable instincts you'll build this semester. It shows up again in Pandas starting Lecture 3, and in every ML model you train starting Week 6.